In [1]:
import re
import pandas as pd

# ---------------------- НОРМАЛИЗАЦИЯ НАЗВАНИЙ ----------------------
exercise_mapping = {
    'горизонтальная тяга в блоке' : 'горизонтальная тяга',
    'горизонтальна тяга в блоке макс вес' : 'горизонтальная тяга',
    'dips': 'брусья',
    'pull-ups': 'подтягивания',
    'pull ups': 'подтягивания',
    'подтягивания вертикальным хватом': 'подтягивания',
    'подтягивания параллельным хватом': 'подтягивания',
    'взрывные pull ups с паузой и медленный опусканием': 'подтягивания',
    'squats': 'присед',
    'smith machine squats': 'присед',
    'присяд': 'присед',
    'легкий присед': 'присед',
    'bench press': 'жим лёжа',
    'жим': 'жим лёжа',
    'dumbbell lateral raise': 'махи в стороны',
    'махи с гантелями': 'махи в стороны',
    'махи': 'махи в стороны',
    'biceps curls': 'бицепс',
    'biceps': 'бицепс',
    'бицепс': 'бицепс',
    'biceps молотки': 'бицепс молотковые',
    'бицепс молотки': 'бицепс молотковые',
    'бицепс с грифом': 'бицепс',
    'бицепс с гантелями': 'бицепс',
    'overhead extensions': 'трицепс',
    'трицепс': 'трицепс',
    'трицепс с гантелью': 'трицепс',
    'трицепс гантель': 'трицепс',
    'трицепс в блоке': 'трицепс',
    'трицепс блок': 'трицепс',
    'разгибания в блоке на трицепс прямая рукоять': 'трицепс',
    'разгибания в блоке на трицепс канат': 'трицепс',
    'разгибания в блоке': 'трицепс',
    'разгибания гантели': 'трицепс',
    'разгибания с гантелью из-за головы': 'трицепс',
    'разгибания на трицепс в блоке прямой рукоятью': 'трицепс',
    'dumbbell shrugs': 'шраги',
    'shrugs': 'шраги',
    'шраги': 'шраги',
    'шраги с гантелями': 'шраги',
    'и без перерыва шраги': 'шраги',
    'скручивания': 'скручивания',
    'скручивания супер сет': 'скручивания',
    'велотренажер': 'велотренажёр',
}

def normalize_exercise(name):
    name = name.lower().strip()
    return exercise_mapping.get(name, name)

# ---------------------- АВТООПРЕДЕЛЕНИЕ ТИПА ПО ВЕСУ ----------------------
def fix_exercise_by_weight(exercise, weight):
    if weight is None:
        return exercise
    if 'бицепс' in exercise:
        return 'бицепс со штангой' if weight > 25 else 'бицепс с гантелями'
    if 'трицепс' in exercise:
        return 'трицепс в блоке' if weight > 25 else 'трицепс с гантелью'
    return exercise

# ---------------------- ПАРСЕР ПОДХОДОВ ----------------------
def parse_sets(weight, reps_str):
    """Превращает '12-10-8' или '20-18' в список подходов (любое количество)"""
    reps_list = [int(r) for r in reps_str.split('-') if r.strip()]
    return [(weight, r) for r in reps_list]

# ---------------------- ПАРСЕР УПРАЖНЕНИЙ ----------------------
def parse_exercise(line):
    line = line.strip()
    if not line:
        return None
    
    # НОВЫЙ ШАБЛОН 1: Вес с запятой и двоеточием
    match = re.match(r'([A-Za-zА-Яа-я\s]+):\s*(\d+[.,]?\d*):\s*([\d\-]+)', line)
    if match:
        name, weight_str, reps_str = match.groups()
        weight = float(weight_str.replace(',', '.'))
        sets = parse_sets(weight, reps_str)
        return {'name': name.strip().lower(), 'sets': sets}
    
    # НОВЫЙ ШАБЛОН 2: Целый вес со звёздочкой (с пробелом или без)
    match = re.match(r'([A-Za-zА-Яа-я\s]+):\s*(\d+)\*\s*([\d\-]+)', line)
    if match:
        name, weight_str, reps_str = match.groups()
        weight = int(weight_str)
        sets = parse_sets(weight, reps_str)
        return {'name': name.strip().lower(), 'sets': sets}
    
    # 0) Смена веса в подходах (только если есть ЧИСЛО-ЧИСЛО-ЧИСЛО*ЧИСЛО)
    if re.search(r'\d+\*\d+-\d+-\d+\*\d+', line):
        match = re.match(r'([A-Za-zА-Яа-я\s\-]+):\s*(.+)', line)
        if match:
            name, rest = match.groups()
            parts = rest.split('-')
            sets = []
            current_weight = None
            for part in parts:
                part = part.strip()
                if '*' in part:
                    w_part, r_part = part.split('*', 1)
                    w_clean = re.search(r'(\d+(?:[.,]\d+)?)', w_part)
                    if w_clean:
                        weight_str = w_clean.group(1).replace(',', '.')
                        current_weight = float(weight_str)
                        r_clean = re.search(r'\d+', r_part)
                        if r_clean:
                            reps = int(r_clean.group())
                            sets.append((current_weight, reps))
                else:
                    if current_weight is not None and part.isdigit():
                        sets.append((current_weight, int(part)))
            if sets:
                return {'name': name.strip().lower(), 'sets': sets}
    
    # 1) Разные веса на подходы: Smith Machine squats: 50*12 - 60*10 - 60*8
    match = re.match(r'([A-Za-zА-Яа-я\s\-]+):\s*(\d+\*\d+(?:\s*-\s*\d+\*\d+)+)', line)
    if match:
        name, sets_str = match.groups()
        parts = re.findall(r'(\d+)\*(\d+)', sets_str)
        sets = [(int(w), int(r)) for w, r in parts]
        return {'name': name.strip().lower(), 'sets': sets}
    
    # 2) Вес с запятой или точкой и звёздочкой: Бицепс: 17,5*10-8-6
    match = re.match(r'([A-Za-zА-Яа-я\s]+):\s*(\d+[.,]?\d*)\*([\d\-]+)', line)
    if match:
        name, weight_str, reps_str = match.groups()
        weight = float(weight_str.replace(',', '.'))
        sets = parse_sets(weight, reps_str)
        return {'name': name.strip().lower(), 'sets': sets}
    
    # 3) Целый вес и звёздочка: Pull-ups: 5*12-10-8  или Pull ups: 5*12-10-8
    match = re.match(r'([A-Za-zА-Яа-я\s\-]+):\s*(\d+)\*([\d\-]+)', line)
    if match:
        name, weight_str, reps_str = match.groups()
        weight = int(weight_str)
        sets = parse_sets(weight, reps_str)
        return {'name': name.strip().lower(), 'sets': sets}
    
    # 4) Вес через двоеточие: Разгибания: 50: 12-10-8
    match = re.match(r'([A-Za-zА-Яа-я\s]+):\s*(\d+):\s*([\d\-]+)', line)
    if match:
        name, weight_str, reps_str = match.groups()
        weight = int(weight_str)
        sets = parse_sets(weight, reps_str)
        return {'name': name.strip().lower(), 'sets': sets}
    
    # 5) Только повторы (без веса): Dips: 20-18-20
    match = re.match(r'([A-Za-zА-Яа-я\s\-]+):\s*([\d\-]+)', line)
    if match:
        name, reps_str = match.groups()
        sets = parse_sets(None, reps_str)
        return {'name': name.strip().lower(), 'sets': sets}
    
    return None

# ---------------------- ПАРСЕР ДАТ ----------------------
def parse_date_with_year(line, last_year):
    match = re.search(r'(\d{2})\.(\d{2})\.(\d{2})', line)
    if match:
        day, month, short_year = match.groups()
        year = 2000 + int(short_year)
        if year < last_year:
            year += 1
        return f"{year}-{month}-{day}", year
    return None, last_year

# ---------------------- ОСНОВНАЯ ФУНКЦИЯ ----------------------
def parse_log(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    
    records = []
    current_date = None
    current_year = 2025
    
    for line in lines:
        line = line.strip()
        if not line:
            continue
        
        date_str, current_year = parse_date_with_year(line, current_year)
        if date_str:
            current_date = date_str
            continue
        
        if current_date:
            ex = parse_exercise(line)
            if ex:
                for weight, reps in ex['sets']:
                    records.append({
                        'date': current_date,
                        'exercise': ex['name'],
                        'weight_kg': weight,
                        'reps': reps,
                        'raw_line': line
                    })
    
    return pd.DataFrame(records)

# ---------------------- ЗАПУСК ----------------------
if __name__ == "__main__":
    df = parse_log('/Users/fliisii/Downloads/text.txt')
    
    # Нормализация названий
    df['exercise'] = df['exercise'].apply(normalize_exercise)
    
    # Автоопределение типа бицепса и трицепса
    df['exercise'] = df.apply(lambda row: fix_exercise_by_weight(row['exercise'], row['weight_kg']), axis=1)
    
    # Преобразуем даты
    df['date'] = pd.to_datetime(df['date']).dt.date
    
    # Исправляем ошибочную дату 2026-12-30 → 2025-12-30
    df.loc[df['date'] == pd.Timestamp('2026-12-30').date(), 'date'] = pd.Timestamp('2025-12-30').date()
    
    # Сохраняем
    df.to_csv('training_dataset', index=False, encoding='utf-8-sig')
    
    print(f"✅ Обработано {len(df)} подходов")
    print(f"📅 Период: {df['date'].min()} → {df['date'].max()}")
    print(f"🏋️ Уникальных упражнений: {df['exercise'].nunique()}")
    print("\nПервые 10 строк:")
    print(df.head(10))

/opt/anaconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


✅ Обработано 1051 подходов
📅 Период: 2025-09-14 → 2026-12-19
🏋️ Уникальных упражнений: 18

Первые 10 строк:
         date        exercise  weight_kg  reps  \
0  2025-09-14          брусья        NaN    20   
1  2025-09-14          брусья        NaN    18   
2  2025-09-14          брусья        NaN    20   
3  2025-09-14    подтягивания        NaN    16   
4  2025-09-14    подтягивания        NaN    14   
5  2025-09-14    подтягивания        NaN    12   
6  2025-09-14          присед       50.0    12   
7  2025-09-14  махи в стороны       12.5    16   
8  2025-09-14  махи в стороны       12.5    14   
9  2025-09-14  махи в стороны       12.5    12   

                                     raw_line  
0                              Dips: 20-18-20  
1                              Dips: 20-18-20  
2                              Dips: 20-18-20  
3                          Pull-ups: 16-14-12  
4                          Pull-ups: 16-14-12  
5                          Pull-ups: 16-14-12  
6  Sm

In [2]:
# Удаляем все строки с reps > 50
df = df[df['reps'] <= 50]

# Сбрасываем индексы
df = df.reset_index(drop=True)

print(f"После удаления аномалий: {len(df)} строк")
print(f"Максимальное значение reps: {df['reps'].max()}")
print(f"Строк с reps > 30: {len(df[df['reps'] > 30])}")

# Сохраняем в CSV
df.to_csv('training_data_clean_final.csv', index=False, encoding='utf-8-sig')
print("✅ Сохранён файл: training_data_clean_final.csv")

После удаления аномалий: 1039 строк
Максимальное значение reps: 50
Строк с reps > 30: 10
✅ Сохранён файл: training_data_clean_final.csv


In [2]:
# Удаляем строки с reps > 34
df = df[df['reps'] <= 34]

# Сбрасываем индексы
df = df.reset_index(drop=True)

print(f"После удаления: {len(df)} строк")
print(f"Максимальное значение reps: {df['reps'].max()}")

После удаления: 1031 строк
Максимальное значение reps: 32


In [3]:
import pandas as pd

# Загружаем данные
df = pd.read_csv('training_data_clean_final.csv', parse_dates=['date'])

print("=" * 60)
print("ПРОВЕРКА АНОМАЛИЙ В ДАННЫХ")
print("=" * 60)

# 1. Повторения > 34
print(f"\n1. Повторения > 34: {len(df[df['reps'] > 34])}")
if len(df[df['reps'] > 34]) > 0:
    print(df[df['reps'] > 34][['date', 'exercise', 'reps', 'raw_line']].head(10))

# 2. Повторения = 0
print(f"\n2. Повторения = 0: {len(df[df['reps'] == 0])}")

# 3. Вес > 200 кг (нереалистично)
print(f"\n3. Вес > 200 кг: {len(df[(df['weight_kg'] > 200) & (df['weight_kg'].notna())])}")

# 4. Вес < 0 (отрицательный)
print(f"\n4. Вес < 0: {len(df[df['weight_kg'] < 0])}")

# 5. Вес есть, но повторений нет
print(f"\n5. Вес есть, повторений нет: {len(df[(df['weight_kg'].notna()) & (df['reps'].isna())])}")

# 6. В raw_line есть '*', но вес не распарсился
has_star_no_weight = df[df['raw_line'].str.contains('\*', na=False) & df['weight_kg'].isna()]
print(f"\n6. Есть '*', но вес не распарсился: {len(has_star_no_weight)}")
if len(has_star_no_weight) > 0:
    print(has_star_no_weight[['date', 'exercise', 'raw_line']].head(10))

# 7. Вес и повторения одинаковые (>20) — явная путаница
weight_eq_reps = df[(df['weight_kg'] == df['reps']) & (df['weight_kg'] > 20) & (df['weight_kg'].notna())]
print(f"\n7. Вес = повторениям (>20): {len(weight_eq_reps)}")
if len(weight_eq_reps) > 0:
    print(weight_eq_reps[['date', 'exercise', 'weight_kg', 'reps', 'raw_line']])

# 8. Проверка на выбросы по весу для конкретных упражнений
print("\n8. Максимальные веса по упражнениям:")
exercises = ['присед', 'жим лёжа', 'жим ногами', 'бицепс с гантелями', 'бицепс со штангой']
for ex in exercises:
    max_weight = df[df['exercise'] == ex]['weight_kg'].max()
    if pd.notna(max_weight):
        print(f"   {ex}: {max_weight} кг")

# 9. Пустые строки или NaN в важных колонках
print(f"\n9. Пустые даты: {df['date'].isna().sum()}")
print(f"   Пустые упражнения: {df['exercise'].isna().sum()}")
print(f"   Пустые повторения: {df['reps'].isna().sum()}")

# 10. Дубликаты строк (полностью идентичные)
duplicates = df.duplicated(subset=['date', 'exercise', 'weight_kg', 'reps'], keep=False)
print(f"\n10. Потенциальные дубликаты: {duplicates.sum()}")

print("\n" + "=" * 60)
print("ИТОГО:")
print(f"Всего строк: {len(df)}")
print(f"Уникальных упражнений: {df['exercise'].nunique()}")
print(f"Диапазон дат: {df['date'].min()} → {df['date'].max()}")

ПРОВЕРКА АНОМАЛИЙ В ДАННЫХ

1. Повторения > 34: 0

2. Повторения = 0: 0

3. Вес > 200 кг: 0

4. Вес < 0: 0

5. Вес есть, повторений нет: 0

6. Есть '*', но вес не распарсился: 0

7. Вес = повторениям (>20): 0

8. Максимальные веса по упражнениям:
   присед: 120.0 кг
   жим лёжа: 60.0 кг
   жим ногами: 150.0 кг
   бицепс с гантелями: 17.5 кг
   бицепс со штангой: 30.0 кг

9. Пустые даты: 0
   Пустые упражнения: 0
   Пустые повторения: 0

10. Потенциальные дубликаты: 92

ИТОГО:
Всего строк: 1028
Уникальных упражнений: 20
Диапазон дат: 2025-09-14 00:00:00 → 2026-12-19 00:00:00


<>:28: SyntaxWarning: invalid escape sequence '\*'
<>:28: SyntaxWarning: invalid escape sequence '\*'
/var/folders/04/1plqmt7n64v5gh88mv_1j2v80000gn/T/ipykernel_6731/1194822170.py:28: SyntaxWarning: invalid escape sequence '\*'
  has_star_no_weight = df[df['raw_line'].str.contains('\*', na=False) & df['weight_kg'].isna()]


In [5]:
# Находим полные дубликаты (все колонки одинаковые)
duplicates = df[df.duplicated(subset=['date', 'exercise', 'weight_kg', 'reps'], keep=False)]

# Сортируем для удобства
duplicates = duplicates.sort_values(['date', 'exercise'])

print(f"Найдено строк-дубликатов: {len(duplicates)}")
print("\nПервые 20 дубликатов:")
print(duplicates[['date', 'exercise', 'weight_kg', 'reps', 'raw_line']].head(20))

# Покажем, как они выглядят в группах
print("\nГруппировка дубликатов:")
dup_groups = duplicates.groupby(['date', 'exercise', 'weight_kg', 'reps']).size().reset_index(name='count')
print(dup_groups[dup_groups['count'] > 1].head(10))

Найдено строк-дубликатов: 92

Первые 20 дубликатов:
          date            exercise  weight_kg  reps  \
0   2025-09-14              брусья        NaN    20   
2   2025-09-14              брусья        NaN    20   
56  2025-09-23      махи в стороны       12.5    14   
58  2025-09-23      махи в стороны       12.5    14   
139 2025-10-12      махи в стороны       10.0    16   
140 2025-10-12      махи в стороны       10.0    16   
141 2025-10-12      махи в стороны       10.0    16   
158 2025-10-14              присед       60.0    10   
159 2025-10-14              присед       60.0    10   
218 2025-10-28      махи в стороны       12.5    14   
219 2025-10-28      махи в стороны       12.5    14   
249 2025-11-01      махи в стороны       12.5    14   
250 2025-11-01      махи в стороны       12.5    14   
282 2025-11-05  трицепс с гантелью       17.5    12   
283 2025-11-05  трицепс с гантелью       17.5    12   
372 2025-11-27            жим лёжа       52.5    10   
373 2025-11-2

In [4]:
import pandas as pd

# Загружаем данные (если ещё не загружены)
df = pd.read_csv('training_data_clean_final.csv', parse_dates=['date'])

print(f"Было строк: {len(df)}")

# 1. Удаляем строки с reps > 34
df = df[df['reps'] <= 34]

# 2. Удаляем строки, где есть '*', но вес не распарсился (это явная ошибка)
df = df[~((df['raw_line'].str.contains(r'\*', na=False)) & (df['weight_kg'].isna()))]

# Сбрасываем индексы
df = df.reset_index(drop=True)

print(f"Стало строк: {len(df)}")

# Сохраняем
df.to_csv('training_data_clean_final.csv', index=False, encoding='utf-8-sig')
print("✅ Сохранён training_data_clean_final.csv")

Было строк: 1028
Стало строк: 1028
✅ Сохранён training_data_clean_final.csv


In [5]:
df['exercise'].unique()

<ArrowStringArray>
[                             'брусья',                        'подтягивания',
                              'присед',                      'махи в стороны',
                  'трицепс с гантелью',                  'бицепс с гантелями',
                               'шраги',                     'трицепс в блоке',
         'горизонтальная тяга в блоке',                 'горизонтальная тяга',
                             'пуловер',                         'скручивания',
                            'жим лёжа', 'горизонтальна тяга в блоке макс вес',
          'жим гантелей сидя на плечи',                'тяга гантели к поясу',
                   'бицепс со штангой',                          'жим ногами',
            'тяга вертикального блока',                        'велотренажёр']
Length: 20, dtype: str

In [6]:
df[df['exercise'] == 'горизонтальна тяга в блоке макс вес']

,date,exercise,weight_kg,reps,raw_line
227,2025-10-30,горизонтальна тяга в блоке макс вес,NaN,16,Горизонтальна тяга в блоке макс вес: 16-14-12
228,2025-10-30,горизонтальна тяга в блоке макс вес,NaN,14,Горизонтальна тяга в блоке макс вес: 16-14-12
229,2025-10-30,горизонтальна тяга в блоке макс вес,NaN,12,Горизонтальна тяга в блоке макс вес: 16-14-12


In [7]:
# Добавляем вес 80 для горизонтальной тяги с макс весом
mask = df['exercise'] == 'горизонтальна тяга в блоке макс вес'
df.loc[mask, 'weight_kg'] = 80.0

# Проверяем, что изменилось
print(df[mask][['date', 'exercise', 'weight_kg', 'reps']].head())

          date                             exercise  weight_kg  reps
227 2025-10-30  горизонтальна тяга в блоке макс вес       80.0    16
228 2025-10-30  горизонтальна тяга в блоке макс вес       80.0    14
229 2025-10-30  горизонтальна тяга в блоке макс вес       80.0    12


In [8]:
df.to_csv('training_data_clean_final.csv', index=False, encoding='utf-8-sig')
print("✅ Сохранено")

✅ Сохранено


In [9]:
# Загружаем финальный файл
df = pd.read_csv('training_data_clean_final.csv')

# Применяем нормализацию
df['exercise'] = df['exercise'].apply(normalize_exercise)

# Проверяем уникальные значения
print(df['exercise'].unique())

# Сохраняем
df.to_csv('training_data_clean_final.csv', index=False, encoding='utf-8-sig')

<ArrowStringArray>
[                    'брусья',               'подтягивания',
                     'присед',             'махи в стороны',
                    'трицепс',                     'бицепс',
                      'шраги',        'горизонтальная тяга',
                    'пуловер',                'скручивания',
                   'жим лёжа', 'жим гантелей сидя на плечи',
       'тяга гантели к поясу',          'бицепс со штангой',
                 'жим ногами',   'тяга вертикального блока',
               'велотренажёр']
Length: 17, dtype: str


In [10]:
import pandas as pd

df = pd.read_csv('training_data_clean_final.csv', parse_dates=['date'])

print("=== ФИНАЛЬНАЯ ПРОВЕРКА ДАННЫХ ===\n")
print(f"Всего строк: {len(df)}")
print(f"Уникальных упражнений: {df['exercise'].nunique()}")
print(f"Диапазон дат: {df['date'].min().date()} → {df['date'].max().date()}")
print(f"\nПовторения > 34: {len(df[df['reps'] > 34])}")
print(f"Вес > 200 кг: {len(df[(df['weight_kg'] > 200) & (df['weight_kg'].notna())])}")
print(f"Пустые даты: {df['date'].isna().sum()}")
print(f"Пустые повторения: {df['reps'].isna().sum()}")
print(f"Вес с '*' не распарсился: {len(df[df['raw_line'].str.contains(r'\*', na=False) & df['weight_kg'].isna()])}")
print(f"\n✅ Данные готовы к загрузке в DataLens")

=== ФИНАЛЬНАЯ ПРОВЕРКА ДАННЫХ ===

Всего строк: 1028
Уникальных упражнений: 17
Диапазон дат: 2025-09-14 → 2026-12-19

Повторения > 34: 0
Вес > 200 кг: 0
Пустые даты: 0
Пустые повторения: 0
Вес с '*' не распарсился: 0

✅ Данные готовы к загрузке в DataLens


In [12]:
from datetime import date

# Словарь замен: старый → новый (сравниваем с date, присваиваем date)
date_mapping = {
    date(2026, 12, 11): date(2025, 12, 11),
    date(2026, 12, 15): date(2025, 12, 15),
    date(2026, 12, 19): date(2025, 12, 19),
    date(2026, 12, 30): date(2025, 12, 30),
}

# Заменяем
for old, new in date_mapping.items():
    df.loc[df['date'] == pd.Timestamp(old), 'date'] = pd.Timestamp(new)

In [13]:
print("Даты после замены:")
print(df['date'].unique())

Даты после замены:
<DatetimeArray>
['2025-09-14 00:00:00', '2025-09-17 00:00:00', '2025-09-20 00:00:00',
 '2025-09-23 00:00:00', '2025-10-02 00:00:00', '2025-10-05 00:00:00',
 '2025-10-07 00:00:00', '2025-10-09 00:00:00', '2025-10-12 00:00:00',
 '2025-10-14 00:00:00', '2025-10-18 00:00:00', '2025-10-25 00:00:00',
 '2025-10-28 00:00:00', '2025-10-30 00:00:00', '2025-11-01 00:00:00',
 '2025-11-03 00:00:00', '2025-11-05 00:00:00', '2025-11-07 00:00:00',
 '2025-11-10 00:00:00', '2025-11-18 00:00:00', '2025-11-21 00:00:00',
 '2025-11-27 00:00:00', '2025-11-30 00:00:00', '2025-12-07 00:00:00',
 '2025-12-11 00:00:00', '2025-12-15 00:00:00', '2025-12-19 00:00:00',
 '2025-12-30 00:00:00', '2026-01-05 00:00:00', '2026-01-09 00:00:00',
 '2026-01-13 00:00:00', '2026-01-23 00:00:00', '2026-01-28 00:00:00',
 '2026-02-01 00:00:00', '2026-02-03 00:00:00', '2026-02-09 00:00:00',
 '2026-02-11 00:00:00', '2026-02-13 00:00:00', '2026-02-15 00:00:00',
 '2026-02-17 00:00:00', '2026-02-19 00:00:00', '2026-02

In [14]:
df.to_csv('training_data_clean_final.csv', index=False, encoding='utf-8-sig')
print("✅ Сохранено")

✅ Сохранено


In [16]:
df['date'].unique()

<DatetimeArray>
['2025-09-14 00:00:00', '2025-09-17 00:00:00', '2025-09-20 00:00:00',
 '2025-09-23 00:00:00', '2025-10-02 00:00:00', '2025-10-05 00:00:00',
 '2025-10-07 00:00:00', '2025-10-09 00:00:00', '2025-10-12 00:00:00',
 '2025-10-14 00:00:00', '2025-10-18 00:00:00', '2025-10-25 00:00:00',
 '2025-10-28 00:00:00', '2025-10-30 00:00:00', '2025-11-01 00:00:00',
 '2025-11-03 00:00:00', '2025-11-05 00:00:00', '2025-11-07 00:00:00',
 '2025-11-10 00:00:00', '2025-11-18 00:00:00', '2025-11-21 00:00:00',
 '2025-11-27 00:00:00', '2025-11-30 00:00:00', '2025-12-07 00:00:00',
 '2025-12-11 00:00:00', '2025-12-15 00:00:00', '2025-12-19 00:00:00',
 '2025-12-30 00:00:00', '2026-01-05 00:00:00', '2026-01-09 00:00:00',
 '2026-01-13 00:00:00', '2026-01-23 00:00:00', '2026-01-28 00:00:00',
 '2026-02-01 00:00:00', '2026-02-03 00:00:00', '2026-02-09 00:00:00',
 '2026-02-11 00:00:00', '2026-02-13 00:00:00', '2026-02-15 00:00:00',
 '2026-02-17 00:00:00', '2026-02-19 00:00:00', '2026-02-21 00:00:00',
 '20